# Multiple linear regression from scratch with the normal equation

Diabetes data: 442 patients, 10 inputs, target = disease progression one year later.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [ ]:
X, y = load_diabetes(return_X_y=True)
print(X.shape, y.shape)          # (442, 10) (442,)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

## scikit-learn first

In [ ]:
reg = LinearRegression().fit(X_train, y_train)
print(r2_score(y_test, reg.predict(X_test)))
print(reg.coef_.round(2), round(reg.intercept_, 2))

## Our own class

In [ ]:
class MyMLR:
    def fit(self, X, y):
        # add the column of 1s for the intercept, at position 0
        Xb = np.insert(X, 0, 1, axis=1)
        # normal equation: beta = (X^T X)^-1 X^T y
        betas = np.linalg.inv(Xb.T @ Xb) @ Xb.T @ y
        self.intercept_ = betas[0]
        self.coef_ = betas[1:]
        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_

mine = MyMLR().fit(X_train, y_train)
print(r2_score(y_test, mine.predict(X_test)))
print(mine.coef_.round(2), round(mine.intercept_, 2))

In [ ]:
# largest difference from scikit-learn: tiny rounding only
np.abs(mine.coef_ - reg.coef_).max()

## A safer way to solve: no explicit inverse

In [ ]:
Xb = np.insert(X_train, 0, 1, axis=1)
b_solve = np.linalg.solve(Xb.T @ Xb, Xb.T @ y_train)      # solves X^T X beta = X^T y directly
b_lstsq = np.linalg.lstsq(Xb, y_train, rcond=None)[0]     # least squares, what libraries use
print(np.abs(b_solve[1:] - reg.coef_).max(), np.abs(b_lstsq[1:] - reg.coef_).max())